# US 3.1 -- Multi-Modal Data Collection

Before we can train a CycleGAN to translate between AOI and USM modalities,
we need to prepare our data.  CycleGAN is designed for **unpaired** translation --
we do not need pixel-aligned pairs of AOI and USM images.  However, for
**evaluation** it is useful to have a small paired set where we know the
correspondence.

This notebook covers:

1. Data folder structure for unpaired training
2. Loading data with `UnpairedDataset` and `PairedDataset`
3. Visualising samples from both modalities
4. The `udm-epic3 prepare` CLI command

## Prerequisites

- Python 3.12, PyTorch
- Install: `uv pip install -e ".[epic3]"`

---
## 1. Data Folder Structure

CycleGAN expects two separate folders -- one for each domain.  The images
do **not** need to be paired or have matching filenames.

```
data/
  epic3/
    trainA/          # AOI training images (domain A)
      aoi_001.png
      aoi_002.png
      ...
    trainB/          # USM training images (domain B)
      usm_001.png
      usm_002.png
      ...
    testA/           # AOI test images
      ...
    testB/           # USM test images
      ...
    paired/          # (optional) paired evaluation set
      aoi/
        sample_001.png
      usm/
        sample_001.png   # same filename = same subject
      masks/
        sample_001.png   # defect mask for evaluation
```

The unpaired sets can have different numbers of images.  The dataloader
cycles through the shorter set to match the longer one.

---
## 2. Loading Data with UnpairedDataset

In [ ]:
import torch
from pathlib import Path
from torchvision import transforms

from udm_epic3.data.unpaired_dataset import UnpairedDataset, PairedDataset

In [ ]:
# Define transforms for both modalities
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # -> [-1, 1]
])

# Create the unpaired dataset
# NOTE: replace these paths with your actual data paths
data_root = Path("data/epic3")
dataset = UnpairedDataset(
    root_A=data_root / "trainA",   # AOI images
    root_B=data_root / "trainB",   # USM images
    transform_A=transform,
    transform_B=transform,
)

print(f"UnpairedDataset created:")
print(f"  Domain A (AOI) images: {len(dataset.paths_A)}")
print(f"  Domain B (USM) images: {len(dataset.paths_B)}")
print(f"  Effective length:      {len(dataset)} (max of both)")

In [ ]:
# Fetch a sample -- returns a dict with 'A' and 'B' tensors
sample = dataset[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"  A (AOI) shape: {sample['A'].shape}, range: [{sample['A'].min():.2f}, {sample['A'].max():.2f}]")
print(f"  B (USM) shape: {sample['B'].shape}, range: [{sample['B'].min():.2f}, {sample['B'].max():.2f}]")
print(f"\nNote: A and B are NOT paired -- they are random independent samples.")

---
## 3. Paired Dataset for Evaluation

For quantitative evaluation (SSIM, Dice score), we use a small set of paired
images where the same subject was imaged with both modalities.

In [ ]:
# Create a paired evaluation dataset
paired_dataset = PairedDataset(
    root_A=data_root / "paired" / "aoi",
    root_B=data_root / "paired" / "usm",
    root_mask=data_root / "paired" / "masks",  # optional defect masks
    transform=transform,
)

print(f"PairedDataset created:")
print(f"  Paired samples: {len(paired_dataset)}")

# Fetch a paired sample
pair = paired_dataset[0]
print(f"  Keys: {list(pair.keys())}")
print(f"  A shape: {pair['A'].shape}")
print(f"  B shape: {pair['B'].shape}")
print(f"  Mask shape: {pair['mask'].shape}")

---
## 4. Visualising Samples from Both Modalities

Let's look at a few samples from each domain to understand the visual
differences between AOI and USM images.

In [ ]:
import matplotlib.pyplot as plt

def denormalize(tensor):
    """Convert from [-1, 1] back to [0, 1] for display."""
    return (tensor * 0.5 + 0.5).clamp(0, 1)

# Grab 4 unpaired samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    sample = dataset[i]
    
    # AOI (domain A) -- top row
    img_a = denormalize(sample['A']).permute(1, 2, 0).numpy()
    axes[0, i].imshow(img_a)
    axes[0, i].set_title(f"AOI sample {i}")
    axes[0, i].axis("off")
    
    # USM (domain B) -- bottom row
    img_b = denormalize(sample['B']).permute(1, 2, 0).numpy()
    axes[1, i].imshow(img_b)
    axes[1, i].set_title(f"USM sample {i}")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("AOI (Domain A)", fontsize=14)
axes[1, 0].set_ylabel("USM (Domain B)", fontsize=14)
fig.suptitle("Unpaired samples from both modalities", fontsize=16)
fig.tight_layout()
plt.show()

---
## 5. Creating a DataLoader

For training, we wrap the dataset in a standard PyTorch DataLoader.

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

# Fetch one batch
batch = next(iter(train_loader))
print(f"Batch A shape: {batch['A'].shape}")  # [4, 3, 256, 256]
print(f"Batch B shape: {batch['B'].shape}")  # [4, 3, 256, 256]

---
## 6. CLI: `udm-epic3 prepare`

The `prepare` command automates the data preparation steps:

```bash
# Prepare data: scan folders, compute statistics, create train/test splits
udm-epic3 prepare --config configs/epic3_cyclegan.yaml
```

This will:
1. Scan the `trainA` and `trainB` folders for valid images
2. Compute per-channel mean and std for normalization
3. Optionally create a train/val split
4. Write a manifest CSV listing all images and their metadata

The config file specifies paths and preprocessing options:

```yaml
data:
  root: data/epic3
  domain_A: trainA      # AOI
  domain_B: trainB      # USM
  image_size: 256
  augment: true
  paired_eval: paired   # optional paired evaluation folder
```

---
## Next steps

With data prepared, we are ready to train a CycleGAN:

| Notebook | Topic |
|----------|------|
| `epic3_02_training.ipynb` | Baseline CycleGAN training |
| `epic3_03_defect_aware.ipynb` | Defect-aware enhancement |
| `epic3_04_evaluation.ipynb` | Translation quality evaluation |